<a href="https://colab.research.google.com/github/Vaishnavi639/GenAI/blob/main/22610081_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required packages
!pip install transformers faiss-cpu sentence-transformers torch

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Sample document corpus
documents = [
    "The Eiffel Tower is located in Paris, France and was built in 1889.",
    "Python is a high-level programming language known for its simplicity.",
    "Machine learning is a subset of artificial intelligence that uses algorithms.",
    "The Amazon rainforest is the largest tropical rainforest in the world.",
    "Photosynthesis is the process by which plants convert sunlight into energy.",
    "The Internet was invented in the late 1960s as ARPANET.",
    "DNA contains the genetic instructions for all living organisms.",
    "The Great Wall of China is over 13,000 miles long."
]

# Initialize embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Create embeddings and build FAISS index
embeddings = embedding_model.encode(documents)
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))

print(f"Indexed {index.ntotal} documents")

# Load LLM for generation
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token
llm = AutoModelForCausalLM.from_pretrained("distilgpt2")
device = "cuda" if torch.cuda.is_available() else "cpu"
llm.to(device)

# RAG function
def retrieve_and_generate(query, k=2):
    # Retrieve relevant documents
    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(query_embedding.astype('float32'), k)

    retrieved_docs = [documents[idx] for idx in indices[0]]
    context = " ".join(retrieved_docs)

    print(f"\nQuery: {query}")
    print(f"Retrieved Documents: {retrieved_docs}")

    # Generate response with context
    prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(device)

    outputs = llm.generate(
        **inputs,
        max_length=inputs['input_ids'].shape[1] + 100,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        num_return_sequences=1
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.split("Answer:")[-1].strip()

    print(f"Generated Answer: {answer}\n")
    return answer

# Test queries
queries = [
    "Where is the Eiffel Tower?",
    "What is machine learning?",
    "Tell me about photosynthesis"
]

for query in queries:
    retrieve_and_generate(query)

# Add new documents dynamically
new_docs = ["Tokyo is the capital city of Japan with a population over 13 million."]
new_embeddings = embedding_model.encode(new_docs)
index.add(new_embeddings.astype('float32'))
documents.extend(new_docs)

print(f"Total indexed documents: {index.ntotal}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 42.0 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


Query: Where is the Eiffel Tower?
Retrieved Documents: ['The Eiffel Tower is located in Paris, France and was built in 1889.', 'The Great Wall of China is over 13,000 miles long.']
Generated Answer: This is the first building on the eastern half of


Query: What is machine learning?
Retrieved Documents: ['Machine learning is a subset of artificial intelligence that uses algorithms.', 'Python is a high-level programming language known for its simplicity.']
Generated Answer: Machine learning is a subset of artificial intelligence that uses algorithms. Python is a high-level programming language known for its simplicity.
Question:


Query: Tell me about photosynthesis
Retrieved Documents: ['Photosynthesis is the process by which plants convert sunlight into energy.', 'DNA contains the genetic instructions for all living organisms.']
Generated Answer: Photosynthesis is the process by which plants convert sunlight into energy

Total indexed documents: 9
